# Animat exploration

Starting point for analysing evolved animats. Data comes from `animats/<task>/runs/seed_NN/`, produced by
`animats/scripts/evolve.py` and `extract.py` (see CLAUDE.md). Each sampled animat (every 512 generations along
the line of descent) has a TPM, a connectivity matrix and the brain states it visited during the 128 trials.

Node order is MABE's: sensors, motors, hidden (`S0 S1 M0 M1 H0 H1 H2 H3`). TPMs are state-by-node, little-endian.

In [ ]:
import sys
sys.path.insert(0, "../scripts")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from load import list_seeds, all_animats, list_animats, load_animat, lod_data, run_info, to_network

TASK = "task1"
list_seeds(TASK)

## Fitness along each line of descent

In [ ]:
animats = all_animats(TASK)
fig, ax = plt.subplots(figsize=(8, 4))
for seed, g in animats.groupby("seed"):
    ax.plot(g["generation"], g["score"], alpha=0.6, label=f"seed {seed}")
ax.set_xlabel("generation")
ax.set_ylabel("fraction of trials correct")
ax.legend(fontsize="small")
animats.head()

## One animat

In [ ]:
seed = 1
generation = list_animats(TASK, seed)["generation"].max()
a = load_animat(TASK, seed, generation)
labels = a["node_labels"]
print(f"seed {a['seed']}, generation {a['generation']}, score {a['score']:.3f}")
pd.DataFrame(a["cm"], index=labels, columns=labels)   # row -> column

In [ ]:
# states visited during the 128 trials, with their probability of occurrence
pd.DataFrame(a["visited_states"], columns=labels).assign(p=a["visited_probs"]).sort_values("p", ascending=False)

## State-weighted measures

The paper averages each causal measure over all network states the animat experienced during the 128 trials,
weighted by probability of occurrence. `average_over_visited` does that for any function `measure(network, state)`.

Many consecutive animats on a line of descent have identical brains; `animats.csv` has a `tpm_hash` column,
so results can be cached by `(tpm_hash, state)` instead of recomputed.

In [ ]:
def average_over_visited(animat, measure):
    """Probability-weighted mean of measure(network, state) over the animat's visited states."""
    network = to_network(animat)
    values = np.array([measure(network, tuple(int(x) for x in s)) for s in animat["visited_states"]])
    return float(np.sum(values * animat["visited_probs"])), values

In [ ]:
# Example: IIT 3.0 Phi of the major complex, using the pyphi 1.2 API.
# pyphi 2.0 / IIT 4.0 functions are named differently; swap in whichever measure you want here.
import pyphi
pyphi.config.PROGRESS_BARS = False

def phi_max(network, state):
    return pyphi.compute.major_complex(network, state).phi

mean_phi, per_state = average_over_visited(a, phi_max)
mean_phi